In [28]:
# Instalación 

import sys
# Instala en el MISMO intérprete que usa el kernel
!{sys.executable} -m pip install --upgrade google-cloud-bigquery google-cloud-bigquery-storage pandas-gbq pyarrow

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 293 kB 1.4 MB/s            


In [32]:
# comprobación de credenciales

import os, google.auth

# Asegúrate de NO forzar una service account:
if os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"):
    print("Quitando GOOGLE_APPLICATION_CREDENTIALS para usar tus credenciales de usuario…")
    os.environ.pop("GOOGLE_APPLICATION_CREDENTIALS", None)

creds, proj = google.auth.default()
print("🔐 Tipo de credenciales:",
      getattr(creds, "service_account_email", "User creds (no service_account_email)"))
print("🧾 ADC project:", proj)

🔐 Tipo de credenciales: User creds (no service_account_email)
🧾 ADC project: None


In [33]:
# Cliente BigQuery + smoke test (SELECT 1)
from google.cloud import bigquery

PROJECT_ID = "mimic-pruebas"  # tu proyecto de cuota
client = bigquery.Client(project=PROJECT_ID)

# Prueba mínima
test_df = client.query("SELECT 1 AS ok", location="US").to_dataframe()
test_df

E0000 00:00:1760591968.527771    8971 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


,ok
0,1


In [34]:
# primera consulta a MIMIC-IV 
query = """
SELECT COUNT(*) AS pacientes
FROM `physionet-data.mimiciv_3_1_hosp.patients`
"""
df = client.query(query, location="US").to_dataframe()
df

E0000 00:00:1760591974.313600    8971 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


,pacientes
0,364627


In [1]:
#Si alguna vez ves 403 otra vez

#Revisa que estás autenticada como usuario (celda 2 debe imprimir “User creds…”).

#Desde terminal (no notebook):

#gcloud auth application-default login
#gcloud auth application-default set-quota-project mimic-pruebas